# TP 3 — Mini-pipeline Bronze → Silver → Gold
**Durée indicative : 50–70 min**

Objectifs : relier ingestion, qualité, jointure et indicateurs. Ici les trois couches sont des **DataFrames** ; la sauvegarde Parquet est une extension facultative.

> **Environment:** Databricks notebook in a Git folder. Attach a compute resource with PySpark available. Data files are in the adjacent `data/` folder. This lab uses only **synthetic** data.

> **Expected skill level:** first experience with Big Data / Spark. Follow the cells in order.


In [1]:
from pathlib import Path
import csv
from pyspark.sql import functions as F

# Databricks Runtime 14+ sets the current directory to the notebook's folder.
DATA_DIR = Path.cwd().parent / "data"
assert DATA_DIR.is_dir(), f"Data directory not found: {DATA_DIR}. Open the notebook from the Git folder."

def read_small_csv(filename):
    # Python reads the small Git-tracked CSV on the driver; Spark does the processing.
    # For big data in production, read directly from cloud storage instead.
    with (DATA_DIR / filename).open(encoding="utf-8", newline="") as handle:
        rows = list(csv.DictReader(handle))
    return spark.createDataFrame(rows)


## Bronze : garder la donnée brute
D’où vient chaque ligne ? Que ferait-on de la date d’ingestion en production ?

In [2]:
bronze_admissions = read_small_csv("admissions_synthetiques.csv").withColumn("date_ingestion", F.current_date())
bronze_patients = read_small_csv("patients_synthetiques.csv")
bronze_admissions.show(5)

+------------+--------------+-------------+----------+-----------+-------+--------------+
|admission_id|date_admission|duree_minutes|patient_id|    service| statut|date_ingestion|
+------------+--------------+-------------+----------+-----------+-------+--------------+
|        A001|    2026-10-02|          190|      P002|Cardiologie|termine|    2026-11-27|
|        A002|    2026-10-03|           40|      P003| Neurologie|termine|    2026-11-27|
|        A003|    2026-10-04|          190|      P004|  Pediatrie|termine|    2026-11-27|
|        A004|    2026-10-05|           40|      P005|   Urgences|termine|    2026-11-27|
|        A005|    2026-10-06|           90|      P006|Cardiologie|termine|    2026-11-27|
+------------+--------------+-------------+----------+-----------+-------+--------------+
only showing top 5 rows


In [3]:
# Complément : lignage, on garde la trace du fichier d'origine de chaque ligne
# (avec spark.read sur Databricks, on utiliserait la colonne cachée _metadata.file_path)
bronze_admissions = bronze_admissions.withColumn("fichier_source", F.lit("data/admissions_synthetiques.csv"))
bronze_admissions.printSchema()
print("Lignes Bronze :", bronze_admissions.count())

root
 |-- admission_id: string (nullable = true)
 |-- date_admission: string (nullable = true)
 |-- duree_minutes: string (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- service: string (nullable = true)
 |-- statut: string (nullable = true)
 |-- date_ingestion: date (nullable = false)
 |-- fichier_source: string (nullable = false)



Lignes Bronze : 84


**✅ Réponse — d'où vient chaque ligne ? Que faire de la date d'ingestion ?**

- **Origine** : chaque ligne vient du fichier `data/admissions_synthetiques.csv` du dépôt Git, chargé **tel quel** : tout en texte, doublon et erreurs compris (84 lignes). La couche Bronze ne corrige rien, elle garde une copie fidèle de la source pour pouvoir tout recalculer si une règle change. En production, la source serait un export du système d'information hospitalier déposé dans le Data Lake (fichiers, API, flux). La colonne `fichier_source` ajoutée ci-dessus assure le **lignage**.
- **Date d'ingestion** : elle indique *quand la donnée est arrivée*, à ne pas confondre avec `date_admission`, qui est la date métier. En production, elle sert à :
  - **partitionner** la couche Bronze par jour de chargement ;
  - traiter en **incrémental** : seulement les données arrivées depuis la dernière exécution ;
  - **rejouer** un chargement précis après une correction, ou retirer un lot défectueux ;
  - **auditer** (« ce chiffre vient du lot du 27/11 ») et repérer les retards ou les envois en double.

  On stocke en général un horodatage complet (`current_timestamp()`) plutôt qu'une simple date.

## Silver : types et règles qualité
Quel compromis entre rejet et correction ? Réutilisez les règles du TP 2.

In [4]:
silver_admissions = (bronze_admissions
    .withColumn("date_admission", F.to_date("date_admission"))
    .withColumn("duree_minutes", F.col("duree_minutes").cast("int"))
    .dropDuplicates(["admission_id"])
    .filter(F.col("service").isNotNull() & (F.col("service") != "") & (F.col("duree_minutes") >= 0))
    .join(bronze_patients.select("patient_id"), "patient_id", "inner"))
print("Bronze:", bronze_admissions.count(), "Silver:", silver_admissions.count())

Bronze: 84 Silver: 80


In [5]:
# Complément : table de quarantaine Silver, les lignes écartées avec leur motif (règles du TP 2)
from pyspark.sql import Window

admissions_typees = (bronze_admissions
    .withColumn("date_admission", F.to_date("date_admission"))
    .withColumn("duree_minutes", F.col("duree_minutes").cast("int")))
rang = F.row_number().over(Window.partitionBy("admission_id").orderBy("date_ingestion"))

silver_rejets = (admissions_typees
    .withColumn("rang", rang)
    .join(bronze_patients.select("patient_id", F.lit(True).alias("patient_connu")), "patient_id", "left")
    .withColumn("motif_rejet",
        F.when(F.col("rang") > 1, "doublon admission_id")
         .when(F.col("service").isNull() | (F.col("service") == ""), "service manquant")
         .when(F.col("duree_minutes").isNull() | (F.col("duree_minutes") < 0), "durée manquante ou négative")
         .when(F.col("patient_connu").isNull(), "patient inconnu"))
    .filter(F.col("motif_rejet").isNotNull())
    .drop("rang", "patient_connu"))

silver_rejets.select("admission_id", "patient_id", "service", "duree_minutes", "motif_rejet").orderBy("admission_id").show(truncate=False)

+------------+----------+----------+-------------+---------------------------+
|admission_id|patient_id|service   |duree_minutes|motif_rejet                |
+------------+----------+----------+-------------+---------------------------+
|A010        |P011      |Neurologie|240          |doublon admission_id       |
|A081        |P999      |Neurologie|45           |patient inconnu            |
|A082        |P004      |          |90           |service manquant           |
|A083        |P005      |Urgences  |-30          |durée manquante ou négative|
+------------+----------+----------+-------------+---------------------------+



**✅ Réponse — quel compromis entre rejet et correction ?**

- **Corriger** quand la correction est **certaine et documentée** : supprimer des espaces, harmoniser la casse (`cardiologie` → `Cardiologie`), convertir un format de date, supprimer un doublon **exact** (aucune information perdue).
- **Rejeter** (mettre en quarantaine) quand on ne peut pas connaître la bonne valeur : service manquant, durée de −30 min (erreur de signe ? de saisie ? impossible à savoir), patient inconnu. Inventer une valeur fausserait les indicateurs, ce qui est inacceptable sur des données de santé.
- **Le compromis** : un rejet fait perdre de l'information (l'activité est sous-estimée : 4 admissions sur 84 ici) ; une correction risque d'introduire une valeur fausse qui passera inaperçue. On corrige donc seulement ce qui est sûr, on isole le reste avec un **motif**, on **mesure** le taux de rejet et on fait corriger **à la source**, sans jamais rien effacer de la couche Bronze.

Différence avec le TP 2 : ici on déduplique sur la **clé** `admission_id` (`dropDuplicates(["admission_id"])`) et non sur la ligne entière. Le résultat est le même car le doublon `A010` est exact, mais si deux versions **différentes** d'une admission existaient, Spark en garderait une au hasard. Il faudrait alors une règle explicite, par exemple garder la plus récente selon l'horodatage d'ingestion.

## Gold : indicateur de pilotage
Construisez un résultat simple : nombre de passages par date et par service.

In [6]:
gold_activite = (silver_admissions.groupBy("date_admission", "service")
    .agg(F.count("*").alias("nb_admissions"), F.round(F.avg("duree_minutes"), 1).alias("duree_moyenne_min"))
    .orderBy("date_admission", "service"))
gold_activite.show(20)

+--------------+-----------+-------------+-----------------+
|date_admission|    service|nb_admissions|duree_moyenne_min|
+--------------+-----------+-------------+-----------------+
|    2026-10-01|Cardiologie|            1|            120.0|
|    2026-10-01| Neurologie|            1|             25.0|
|    2026-10-01|  Pediatrie|            2|            155.0|
|    2026-10-01|   Urgences|            1|            190.0|
|    2026-10-02|Cardiologie|            2|            155.0|
|    2026-10-02| Neurologie|            1|             25.0|
|    2026-10-02|  Pediatrie|            1|            190.0|
|    2026-10-02|   Urgences|            2|            105.0|
|    2026-10-03|Cardiologie|            2|            180.0|
|    2026-10-03| Neurologie|            2|             52.5|
|    2026-10-03|  Pediatrie|            1|             25.0|
|    2026-10-03|   Urgences|            1|            240.0|
|    2026-10-04|Cardiologie|            1|             40.0|
|    2026-10-04| Neurolo

In [7]:
# Complément : taille de la table Gold et cohérence avec Silver
print("Lignes Gold :", gold_activite.count(),
      "| total des passages :", gold_activite.agg(F.sum("nb_admissions")).first()[0])
gold_activite.groupBy("nb_admissions").count().orderBy("nb_admissions").show()

# Deuxième indicateur Gold : synthèse par service, calculée depuis Silver
# (et non en moyennant les moyennes journalières, ce qui donnerait un résultat faux)
gold_services = (silver_admissions.groupBy("service")
    .agg(F.count("*").alias("nb_admissions"),
         F.countDistinct("patient_id").alias("nb_patients"),
         F.round(F.avg("duree_minutes"), 1).alias("duree_moyenne_min"))
    .orderBy("service"))
gold_services.show()

Lignes Gold : 60 | total des passages : 80


+-------------+-----+
|nb_admissions|count|
+-------------+-----+
|            1|   40|
|            2|   20|
+-------------+-----+



+-----------+-------------+-----------+-----------------+
|    service|nb_admissions|nb_patients|duree_moyenne_min|
+-----------+-------------+-----------+-----------------+
|Cardiologie|           20|         15|            118.8|
| Neurologie|           20|         15|            106.5|
|  Pediatrie|           20|         15|            133.5|
|   Urgences|           20|         15|            129.3|
+-----------+-------------+-----------+-----------------+



**✅ Réponse — l'indicateur Gold**

`gold_activite` compte **60 lignes** : 15 jours (du 1er au 15 octobre 2026) × 4 services, chaque combinaison étant présente, alors que `show(20)` n'en affiche que 20. Chaque combinaison compte 1 ou 2 passages (40 combinaisons à 1, 20 à 2), soit **80 passages au total : exactement le nombre de lignes Silver**. La table Gold est petite, agrégée et ne contient aucune donnée individuelle de patient : elle est prête pour un tableau de bord.

`gold_services` donne la synthèse par service (20 passages chacun, durée moyenne de 106,5 à 133,5 min). Attention : la durée moyenne d'un service sur la période se calcule depuis Silver, **pas** en faisant la moyenne des moyennes journalières, où un jour à 1 passage pèserait autant qu'un jour à 2 passages.

## Contrôles finaux
Un indicateur métier crédible nécessite des règles documentées et une cohérence du nombre de lignes.

In [8]:
assert gold_activite.filter(F.col("nb_admissions") <= 0).count() == 0
assert silver_admissions.filter(F.col("duree_minutes") < 0).count() == 0
print("Contrôles de base OK")

Contrôles de base OK


In [9]:
# Complément : cohérence du nombre de lignes entre les couches
nb_bronze = bronze_admissions.count()
nb_silver = silver_admissions.count()
nb_rejets = silver_rejets.count()
total_gold = gold_activite.agg(F.sum("nb_admissions")).first()[0]

assert nb_bronze == nb_silver + nb_rejets, "des lignes ont disparu sans motif"
assert total_gold == nb_silver, "Gold ne compte pas toutes les admissions Silver"
assert silver_admissions.select("admission_id").distinct().count() == nb_silver, "admission_id non unique en Silver"
assert silver_admissions.filter(F.col("service").isNull() | (F.col("service") == "")).count() == 0, "service manquant en Silver"
assert silver_admissions.join(bronze_patients, "patient_id", "left_anti").count() == 0, "patient inconnu en Silver"

print(f"Bronze {nb_bronze} = Silver {nb_silver} + rejets {nb_rejets} | total Gold = {total_gold} | taux de rejet = {nb_rejets / nb_bronze:.1%}")
print("Contrôles de cohérence OK")

Bronze 84 = Silver 80 + rejets 4 | total Gold = 80 | taux de rejet = 4.8%
Contrôles de cohérence OK


**✅ Règles documentées du pipeline**

| # | Règle | Couche | Traitement | Lignes concernées |
|---|---|---|---|---:|
| 1 | `date_admission` est une date, `duree_minutes` un entier | Silver | conversion de type | 84 |
| 2 | `admission_id` est unique | Silver | doublon écarté | 1 (`A010`) |
| 3 | le service est renseigné | Silver | rejet | 1 (`A082`) |
| 4 | la durée est positive ou nulle | Silver | rejet | 1 (`A083`) |
| 5 | le patient existe dans le référentiel | Silver | rejet | 1 (`A081` → `P999`) |
| 6 | Bronze = Silver + rejets | contrôle | arrêt si faux | 84 = 80 + 4 |
| 7 | total des passages Gold = lignes Silver | contrôle | arrêt si faux | 80 = 80 |

Les `assert` servent de garde-fous : si une règle n'est pas respectée, le notebook s'arrête **avant** de publier un indicateur faux.

## Extension facultative — matérialiser les couches en Parquet
⚠️ Chaque étudiant doit personnaliser `OUTPUT_ROOT`. Il n’est **pas** nécessaire de le faire pour terminer ce TP.

In [10]:
# IMPORTANT: configure one writable Unity Catalog Volume before this step.
# Use a unique subdirectory to avoid overwriting a classmate's output.
import os, re
from pathlib import Path

# Identifiant personnel déduit du compte connecté (ex. "jean.dupont@episen.fr" -> "jean_dupont") :
# chaque étudiant écrit dans son propre sous-dossier, jamais à la racine du volume partagé.
IDENTIFIANT = re.sub(r"[^a-z0-9]+", "_", spark.sql("SELECT current_user()").first()[0].split("@")[0].lower()).strip("_")

if "DATABRICKS_RUNTIME_VERSION" in os.environ or str(Path.cwd()).startswith("/Workspace"):
    VOLUME = "/Volumes/workspace/default/episen_volume"
    assert os.path.isdir(VOLUME), f"Volume introuvable : {VOLUME} (le créer dans Catalog, ou adapter le chemin)"
    OUTPUT_ROOT = f"{VOLUME}/{IDENTIFIANT}"
else:
    # Exécution hors Databricks (Spark installé en local) : dossier output/ du projet, ignoré par Git
    OUTPUT_ROOT = str(Path.cwd().parent / "output" / IDENTIFIANT)

assert "CHANGE_ME" not in OUTPUT_ROOT, "Replace CHANGE_ME with your personal identifier."
print("Dossier de sortie :", OUTPUT_ROOT)

Dossier de sortie : /tmp/EPISEN_TP_1/output/etudiant


In [11]:
bronze_admissions.write.mode("overwrite").parquet(OUTPUT_ROOT + "/bronze_admissions")
silver_admissions.write.mode("overwrite").parquet(OUTPUT_ROOT + "/silver_admissions")
gold_activite.write.mode("overwrite").parquet(OUTPUT_ROOT + "/gold_activite")

In [12]:
# Complément : les rejets sont matérialisés eux aussi, puis on relit chaque couche pour vérifier
silver_rejets.write.mode("overwrite").parquet(OUTPUT_ROOT + "/silver_rejets")
for couche in ["bronze_admissions", "silver_admissions", "silver_rejets", "gold_activite"]:
    print(f"{couche:<18} : {spark.read.parquet(OUTPUT_ROOT + '/' + couche).count():>3} lignes")

bronze_admissions  :  84 lignes
silver_admissions  :  80 lignes


silver_rejets      :   4 lignes
gold_activite      :  60 lignes


### Restitution en 3 minutes
1. Dessiner Sources → Bronze → Silver → Gold → Tableau de bord.
2. Citer deux problèmes de qualité et une règle métier.
3. Expliquer pourquoi le pipeline doit être **orchestré**, **surveillé**, et **relançable**.

**✅ Restitution**

**1. Le pipeline**

```text
  SOURCES          admissions_synthetiques.csv (84 lignes) + patients_synthetiques.csv (31 patients)
     │
     ▼
  BRONZE           copie brute, tout en texte, + date_ingestion et fichier_source        → 84 lignes
     │
     ▼
  SILVER           types (date, int) + règles qualité + jointure au référentiel patients  → 80 lignes
     │      └──▶   table de rejets avec motif                                             →  4 lignes
     ▼
  GOLD             passages par date et par service (gold_activite), synthèse par service → 60 lignes
     │
     ▼
  TABLEAU DE BORD  activité quotidienne et durées moyennes par service
```

**2. Deux problèmes de qualité et une règle métier**
- Problèmes : le **doublon exact** `A010` (la même admission chargée deux fois aurait été comptée deux fois) et le **patient inconnu** `P999` (admission `A081` rattachée à aucun patient du référentiel). Il y avait aussi un service manquant (`A082`) et une durée négative (`A083`).
- Règle métier : **une durée de passage est positive ou nulle** ; une durée négative est rejetée, pas corrigée.

**3. Pourquoi un pipeline orchestré, surveillé et relançable ?**
- **Orchestré** : les étapes dépendent les unes des autres (Silver après Bronze, Gold après Silver) et doivent tourner automatiquement, dans le bon ordre et à intervalle régulier (chaque nuit, par exemple), sans lancer les notebooks à la main. Un orchestrateur (Databricks Jobs, Airflow…) gère les dépendances, la planification et les nouvelles tentatives.
- **Surveillé** : un pipeline peut échouer (fichier absent, format modifié), ou pire, réussir avec des données fausses. On suit le succès ou l'échec et la durée des exécutions, les volumes par couche, le taux de rejet et la fraîcheur des données, avec une alerte au-delà d'un seuil (par exemple un taux de rejet supérieur à 5 % ; il est de 4,8 % ici).
- **Relançable** : après une panne ou une correction de la source, on doit pouvoir relancer sans créer de doublons ni de trous. Il faut des traitements **idempotents** (réécriture `overwrite` d'une partition, `MERGE` sur la clé `admission_id`) et une couche Bronze conservée pour rejouer l'historique.